In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
import ce # type: ignore
import linear # type: ignore
import softmax # type: ignore
import relu # type: ignore
from common import repeat, test_checkgrad_1, test_compare_2, test_model_sgd_1, OneHot # type: ignore
from approx import approx # type: ignore

In [ ]:
class Per_Lin_ReLU_Lin_Softmax_CE_Autograd(nn.Module):
    """The version relying fully on PyTorch to handle `forward` and `backward` passes."""

    def __init__(self, in_features: int, hidden_features: int, out_features: int, w1 = None, b1 = None, w2 = None, b2 = None):
        super().__init__()

        self.linear1 = nn.Linear(in_features, hidden_features)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(hidden_features, out_features)
        self.loss_ = nn.CrossEntropyLoss(reduction='mean')

        if w1 is not None:
            with tr.no_grad():
                self.linear1.weight.copy_(w1)

        if b1 is not None:
            with tr.no_grad():
                self.linear1.bias.copy_(b1)

        if w2 is not None:
            with tr.no_grad():
                self.linear2.weight.copy_(w2)

        if b2 is not None:
            with tr.no_grad():
                self.linear2.bias.copy_(b2)

    def forward(self, x):
        return self.model(x)

    def model(self, x):
        z1 = self.linear1(x)
        a1 = self.relu(z1)
        return self.linear2(a1)

    def loss(self, p, t_labels):
        classes = self.linear2.weight.shape[0]
        t_onehot = OneHot.from_labels(classes)(t_labels.squeeze().long()).float()
        return self.loss_(p, t_onehot)

    def predict(self, x):
        with tr.no_grad():
            return self.model(x).argmax(dim=1)

In [ ]:
class Per_Lin_ReLU_Lin_Softmax_CE_Backward(nn.Module):
    """
    The version where the `forward` and `backward` passes for each operation are written manually, 
    but PyTorch's autograd still orchestrates the overall gradient flow through the computational graph.
    """

    def __init__(self, in_features: int, hidden_features: int, out_features: int, w1 = None, b1 = None, w2 = None, b2 = None):
        super().__init__()

        self.linear1 = linear.Linear(in_features, hidden_features, w1, b1)
        self.relu = relu.ReLU()
        self.linear2 = linear.Linear(hidden_features, out_features, w2, b2)
        self.softmax = softmax.Softmax()
        self.loss_ = ce.CE(reduction='mean')

    def forward(self, x):
        return self.model(x)

    def model(self, x):
        z1 = self.linear1(x)
        a1 = self.relu(z1)
        return self.linear2(a1)

    def loss(self, p, t_labels):
        classes = self.linear2.weight.shape[0]
        t_onehot = OneHot.from_labels(classes)(t_labels.squeeze().long()).float()
        return self.loss_(self.softmax(p), t_onehot)

    def predict(self, x):
        with tr.no_grad():
            return self.model(x).argmax(dim=1)

$$ \text{Definition} $$

$$ z_1 = x \, w_1 + b_1 $$

$$ a = \max(0, z_1) $$

$$ z_2 = a \, w_2 + b_2 $$

$$ p = \text{softmax}(z_2) $$

$$ L = -\sum_i t_i \, \ln(p_i)  $$

$$ F(x, w_1, b_1, w_2, b_2, t) = L(x, w_1, b_1, w_2, b_2, t) $$

$$ \sum_i p_i = 1, \quad \sum_i t_i = 1 $$

$$ \text{Derivatives} $$

$$ \frac{dz_1}{dw_1} = x $$

$$ \frac{dz_1}{db_1} = 1 $$

$$ \frac{da}{dz_1} = \text{diag}(\mathbb{I}_{z_1 > 0}) $$

$$ \frac{dz_2}{dw_2} = a_1 $$

$$ \frac{dz_2}{db_2} = 1 $$

$$ \frac{dp_i}{d(z_2)_j} = p_i(\delta_{ij} - p_j) $$

$$ \frac{dL}{dp_i} = -t_i \frac{1}{p_i} $$

$$ \frac{dF}{dz_2} \overset{Fz_2}{=} p - t $$

$$
Fz_2) \quad \frac{dL}{d(z_2)_j} = 
\sum_i \frac{dL}{dp_i} \frac{dp_i}{d(z_2)_j} = 
\sum_i \Big( -t_i \frac{1}{p_i} \Big) \Big( p_i (\delta_{ij} - p_j) \Big) = 
\sum_i -t_i \, \delta_{ij} + \sum_i t_i \, p_j = 
p_j - t_j
$$

$$ dz_1 = x \, dw_1 + db_1 $$

$$ 
\left \langle \frac{dF}{dw_1}, dw_1 \right \rangle + 
\left \langle \frac{dF}{db_1}, db_1 \right \rangle = 
\left \langle \frac{dF}{dz_1}, dz_1 \right \rangle
$$

$$
\quad \left \langle \frac{dF}{dz_1}, dz_1 \right \rangle = 
\left \langle \frac{dF}{dz_1}, x \, dw_1 + db_1 \right \rangle \implies
$$

$$
\begin{dcases}
    \frac{dF}{dw_1} = x^\top \frac{dF}{dz_1} \\
    \\
    \frac{dF}{db_1} = \frac{dF}{dz_1}
    \end{dcases}
$$

$$ dz_2 = a \, dw_2 + db_2 $$

$$ 
\left \langle \frac{dF}{dw_2}, dw_2 \right \rangle + 
\left \langle \frac{dF}{db_2}, db_2 \right \rangle = 
\left \langle \frac{dF}{dz_2}, dz_2 \right \rangle
$$

$$
\quad \left \langle \frac{dF}{dz_2}, dz_2 \right \rangle = 
\left \langle \frac{dF}{dz_2}, a \, dw_2 + db_2 \right \rangle \implies
$$

$$
\begin{dcases}
    \frac{dF}{dw_2} = a^\top \frac{dF}{dz_2} \\
    \\
    \frac{dF}{db_2} = \frac{dF}{dz_2}
    \end{dcases}
$$

$$
1)
\quad \left \langle \frac{dF}{dz_1}, dz_1 \right \rangle = 
\left \langle \frac{dF}{dz_1}, x \, dw_1 + db_1 \right \rangle = 
$$

$$
\left \langle x^\top \frac{dF}{dz_1}, dw_1 \right \rangle + 
\left \langle \frac{dF}{dz_1}, db_1 \right \rangle \implies
$$

$$ 
\begin{dcases}
    \left \langle \frac{dF}{dw_1}, dw_1 \right \rangle = 
    \left \langle x^\top \frac{dF}{dz_1}, dw_1 \right \rangle \\
    \\
    \left \langle \frac{dF}{db_1}, db_1 \right \rangle = 
    \left \langle \frac{dF}{dz_1}, db_1 \right \rangle
\end{dcases}
$$

$$
\left \langle \frac{dL}{dz_1}, x dw_1 + db_1 \right \rangle =
\left \langle x^\top \frac{dL}{dz_1}, dw_1 \right \rangle + 
\left \langle \frac{dL}{dz_1}, db_1 \right \rangle
$$

$$
2) \quad \left \langle \frac{dF}{dL}, dL \right \rangle =
\frac{dF}{dL} \left \langle \frac{dL}{dz_2}, dz_2 \right \rangle =
$$

$$
\frac{dF}{dL} \left \langle \frac{dL}{dz_2}, (da_1)w_2 + a_1 \, dw_2 + db_2 \right \rangle \overset{w_2, b_2}{=}
$$

$$
\frac{dF}{dL} \left \langle \frac{dL}{dz_2}, a_1 \, dw_2 + db_2 \right \rangle =
$$

$$
\frac{dF}{dL} \left \langle \frac{dL}{dz_2}, a_1 \, dw_2 \right \rangle + 
\frac{dF}{dL} \left \langle \frac{dL}{dz_2}, db_2 \right \rangle =
$$

$$
\frac{dF}{dL} \left \langle {a_1}^\top \, \frac{dL}{dz_2}, dw_2 \right \rangle + 
\frac{dF}{dL} \left \langle \frac{dL}{dz_2}, db_2 \right \rangle \implies
$$

$$
\begin{dcases}
    \left \langle \frac{dF}{dw_2}, dw_2 \right \rangle = \frac{dF}{dL} {a_1}^\top \frac{dL}{dz_2} \\
    \\
    \left \langle \frac{dF}{db_2}, db_2 \right \rangle = \frac{dF}{dL} \frac{dL}{dz_2}
    \end{dcases}
$$

In [ ]:
import torch as tr
import torch.autograd as autograd

class MLP_Softmax_CE_Gradient_Function(autograd.Function):
    
    @staticmethod
    def forward(ctx, x, w1, b1, w2, b2, t_onehot):
        z1 = linear.linear(x, w1, b1)
        a1 = relu.relu(z1)
        z2 = linear.linear(a1, w2, b2)
        p = softmax.softmax(z2)
        L = ce.ce(p, t_onehot)        
        
        ctx.save_for_backward(x, w2, z1, a1, p, t_onehot)
        
        return L

    @staticmethod
    def backward(ctx, dF_dL):
        x, w2, z1, a1, p, t_onehot = ctx.saved_tensors
        S = x.shape[0]
        
        dz2 = (1 / S) * (p - t_onehot)        
        dw2 = a1.t() @ dz2
        db2 = dz2.sum(dim=0)        
        da1 = dz2 @ w2.t()        
        relu_grad = (z1 > 0).float()
        dz1 = da1 * relu_grad
        dw1 = x.t() @ dz1
        dF_db1 = dF_dL * dz1.sum(dim=0)
        
        # # Uwzględnienie zewnętrznego gradientu dF_dL (Chain Rule)
        # dw1 = dF_dL * dw1
        # dF_db1 = dF_db1
        # dw2 = dF_dL * dw2
        # db2 = dF_dL * db2
        
        return None, dw1, dF_db1, dw2, db2, None

In [ ]:
def test_logical_fn(model, bool_fn, samples=100, epochs=500, lr=0.1):
    x = (tr.randn(samples, 2, dtype=tr.float32) > 0).float()
    t = bool_fn(x[:, 0], x[:, 1]).long().unsqueeze(1)
    return test_model_sgd_1(model, x, t, epochs, lr=lr)


if __name__ == "__main__":
    def toOnehot(z: tr.Tensor) -> tr.Tensor:
        return OneHot.from_logits(z.shape[1])(z)

    def toLabel(z: tr.Tensor) -> tr.Tensor:
        return z.argmax(dim=1).float()

    test_compare_2(Per_Lin_ReLU_Lin_Softmax_CE_Autograd, Per_Lin_ReLU_Lin_Softmax_CE_Backward, 2, 3, 4, 5, E=3, fY=toLabel)
    test_compare_2(Per_Lin_ReLU_Lin_Softmax_CE_Autograd, Per_Lin_ReLU_Lin_Softmax_CE_Backward, 10, 20, 30, 40, E=3, fY=toLabel)

    for epochs in [500, 1000]:
        print(f"{epochs}/XOR: {repeat(lambda: test_logical_fn(Per_Lin_ReLU_Lin_Softmax_CE_Autograd(2, 4, 2), tr.logical_xor, epochs=epochs)):.2f}")